In [2]:
# macro_processor_short.py

MNT, MDT, ALA, FORMALS = [], [], {}, {}

def pass1(prog):
    in_macro = False; name = ""
    for line in prog:
        line = line.strip()
        if not line: continue
        if line == "MACRO": in_macro = True; continue
        if in_macro:
            if line == "MEND":
                MDT.append("MEND"); in_macro = False; name = ""; continue
            parts = line.replace(",", " ").split()
            if not name:
                name = parts[0]; MNT.append([len(MNT), name, len(MDT)])
                FORMALS[name] = parts[1:]
            else:
                MDT.append(" ".join(f"#{FORMALS[name].index(t)}" if t in FORMALS[name] else t for t in parts))
        else:
            parts = line.replace(",", " ").split()
            if parts and any(m[1] == parts[0] for m in MNT):
                ALA[parts[0]] = parts[1:]

def pass2(prog):
    expanded, mnt_map = [], {m[1]: m[2] for m in MNT}
    i = 0
    while i < len(prog):
        line = prog[i].strip()
        if not line: i+=1; continue
        if line == "MACRO":
            while prog[i].strip() != "MEND": i+=1
            i+=1; continue
        parts = line.replace(",", " ").split()
        if parts and parts[0] in mnt_map:
            actuals = parts[1:]; ALA[parts[0]] = actuals
            j = mnt_map[parts[0]]
            while MDT[j] != "MEND":
                tokens = [(actuals[int(t[1:])] if t.startswith("#") else t) for t in MDT[j].split()]
                expanded.append(" ".join(tokens)); j+=1
        else: expanded.append(line)
        i+=1
    return expanded

prog = [
    "MACRO","INCR &A,&B","ADD AREG,&A","SUB BREG,&B","MEND",
    "MACRO","DECR &X,&Y","SUB AREG,&X","SUB BREG,&Y","MEND",
    "START 100","INCR A,B","DECR 5,10","END"
]

pass1(prog)
result = pass2(prog)

print("Expanded Program:\n", "\n".join(result))
print("\nMNT:", MNT)
print("MDT:", MDT)
print("FORMALS:", FORMALS)
print("ALA:", ALA)


Expanded Program:
 START 100
ADD AREG A
SUB BREG B
SUB AREG 5
SUB BREG 10
END

MNT: [[0, 'INCR', 0], [1, 'DECR', 3]]
MDT: ['ADD AREG #0', 'SUB BREG #1', 'MEND', 'SUB AREG #0', 'SUB BREG #1', 'MEND']
FORMALS: {'INCR': ['&A', '&B'], 'DECR': ['&X', '&Y']}
ALA: {'INCR': ['A', 'B'], 'DECR': ['5', '10']}
